# TransUNet-RS — Visualization Notebook

This notebook demonstrates:
1. Loading a trained model and running inference
2. Visualizing segmentation predictions
3. Plotting per-class performance metrics
4. Generating confusion matrices
5. Training curve analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.evaluation.visualize import (
    plot_confusion_matrix,
    plot_prediction,
    plot_per_class_metrics,
    plot_training_curves,
    colorize_mask,
    DEFAULT_CLASS_NAMES,
)
from src.evaluation.metrics import SegmentationMetrics

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Simulated Confusion Matrix

In [ ]:
# Generate a realistic-looking confusion matrix
np.random.seed(42)
num_classes = 10
cm = np.random.randint(5, 30, (num_classes, num_classes))
np.fill_diagonal(cm, np.random.randint(300, 600, num_classes))

fig = plot_confusion_matrix(cm, DEFAULT_CLASS_NAMES, normalize=True)
plt.show()

## 2. Per-Class Performance

In [ ]:
# Simulated per-class metrics
per_class_iou = [0.803, 0.912, 0.768, 0.785, 0.821, 0.746, 0.772, 0.854, 0.887, 0.895]
per_class_f1  = [0.891, 0.954, 0.869, 0.880, 0.902, 0.855, 0.871, 0.921, 0.940, 0.945]

fig = plot_per_class_metrics(per_class_iou, per_class_f1, DEFAULT_CLASS_NAMES)
plt.show()

## 3. Training Curves

In [ ]:
# Simulated training history
epochs = 100
train_losses = [1.8 * np.exp(-0.03 * e) + 0.15 + np.random.normal(0, 0.02) for e in range(epochs)]
val_losses = [1.5 * np.exp(-0.025 * e) + 0.20 + np.random.normal(0, 0.03) for e in range(epochs)]
val_mious = [0.82 * (1 - np.exp(-0.04 * e)) + np.random.normal(0, 0.01) for e in range(epochs)]

fig = plot_training_curves(train_losses, val_losses, val_mious)
plt.show()

## 4. Prediction Visualization (Synthetic)

In [ ]:
# Generate a synthetic satellite-like image and masks
H, W = 256, 256
image = np.random.randint(50, 200, (H, W, 3), dtype=np.uint8)
ground_truth = np.random.choice(10, (H, W), p=[0.15, 0.20, 0.10, 0.05, 0.05, 0.10, 0.08, 0.12, 0.07, 0.08])
prediction = ground_truth.copy()
# Add some noise to predictions
noise_mask = np.random.random((H, W)) < 0.05
prediction[noise_mask] = np.random.randint(0, 10, noise_mask.sum())

fig = plot_prediction(image, ground_truth, prediction)
plt.show()

## 5. Metrics Computation

In [ ]:
# Compute metrics using the SegmentationMetrics class
metrics = SegmentationMetrics(num_classes=10)

preds_tensor = torch.from_numpy(prediction).unsqueeze(0)
targets_tensor = torch.from_numpy(ground_truth).unsqueeze(0)

metrics.update(preds_tensor, targets_tensor)
results = metrics.compute()

print('=== Evaluation Results ===')
print(f'Overall Accuracy: {results["oa"]:.4f}')
print(f'Mean IoU:         {results["miou"]:.4f}')
print(f'Macro F1:         {results["f1"]:.4f}')
print(f'Cohen\'s Kappa:   {results["kappa"]:.4f}')
print(f'\nPer-class IoU:')
for name, iou in zip(DEFAULT_CLASS_NAMES, results['per_class_iou']):
    print(f'  {name:>20s}: {iou:.4f}')